# 🎬 Longform Video Factory

Automated faceless whiteboard animation YouTube video pipeline.

**Workflow:**
1. Select topic from Google Sheet
2. Deep research (Gemini)
3. Generate script (Claude/Gemini)
4. ⏸️ **Review script**
5. Generate voiceover (Fish Audio / Qwen3-TTS)
6. Generate scene illustrations (Gemini Flash Image)
7. Assemble video (FFmpeg)
8. ⏸️ **Review video**
9. Generate thumbnail + SEO metadata
10. Save to Google Drive

---
## Cell 0: Setup & Install

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the repo
import os
REPO_DIR = '/content/longform'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/techafreshh/longform.git {REPO_DIR}

# Install dependencies
!pip install -q google-genai fish-audio-sdk openai gspread google-auth \
    python-dotenv requests Pillow openai-whisper

# Add repo to path
import sys
sys.path.insert(0, REPO_DIR)

print('✅ Setup complete!')

In [ ]:
# Configure API keys
# Option A: Set directly (for quick testing)
import os
os.environ['GOOGLE_API_KEY'] = ''       # Your Google AI key
os.environ['OPENROUTER_API_KEY'] = ''    # Your OpenRouter key
os.environ['FISH_API_KEY'] = ''          # Your Fish Audio key
os.environ['FISH_VOICE_ID'] = ''         # Your cloned voice ID
os.environ['PEXELS_API_KEY'] = ''        # Optional: Pexels for B-roll
os.environ['GOOGLE_SHEET_ID'] = ''       # Your Google Sheet ID

# Option B: Load from a .env file on Drive
# from dotenv import load_dotenv
# load_dotenv('/content/drive/MyDrive/LongformFactory/.env')

# Authenticate with Google (for Sheets access)
from google.colab import auth
auth.authenticate_user()

print('✅ API keys configured')

---
## Cell 1: Select Topic

In [ ]:
from src.sheets import get_ready_topics, update_status

# Fetch ready topics from your Google Sheet
topics = get_ready_topics()

if not topics:
    print('⚠️  No topics with status "ready" found in the sheet.')
    print('   Add topics to your sheet or set one manually below.')
else:
    print('\nSelect a topic by number:')
    for i, t in enumerate(topics):
        print(f'  [{i}] {t["topic"]} ({t["niche"]}, {t["style"]})')

In [ ]:
#=== Pick your topic ===#
# Option A: From sheet (set the index from above)
TOPIC_INDEX = 0  # Change this to select a different topic

try:
    selected = topics[TOPIC_INDEX]
    TOPIC = selected['topic']
    NICHE = selected['niche']
    STYLE = selected['style']
    ADDITIONAL_PROMPT = selected['additional_prompt']
    TARGET_LENGTH = selected['target_length']
    SHEET_ROW = selected['row_number']
    print(f'✅ Selected: {TOPIC}')
except (NameError, IndexError):
    # Option B: Manual entry
    TOPIC = 'Why You Can\'t Remember Being a Baby'
    NICHE = 'Psychology'
    STYLE = 'color_whiteboard'  # or 'chalkboard'
    ADDITIONAL_PROMPT = 'Focus on hippocampus development'
    TARGET_LENGTH = '10-12 min'
    SHEET_ROW = None
    print(f'✅ Manual topic: {TOPIC}')

# Update sheet status
if SHEET_ROW:
    update_status(SHEET_ROW, 'researching')

# Output directory on Drive
BASE_DIR = '/content/drive/MyDrive/LongformFactory'
print(f'   Output: {BASE_DIR}/{NICHE}/{TOPIC[:40]}...')

---
## Cell 2: Deep Research

In [ ]:
from src.pipeline import run_pipeline

# Run stages 1-2 (Research + Script)
results = run_pipeline(
    topic=TOPIC,
    niche=NICHE,
    style=STYLE,
    additional_prompt=ADDITIONAL_PROMPT,
    target_length=TARGET_LENGTH,
    base_dir=BASE_DIR,
)

if SHEET_ROW:
    update_status(SHEET_ROW, 'review_script')

print(f'\n📄 {results["scene_count"]} scenes generated')

---
## ⏸️ Cell 3: REVIEW SCRIPT

**Read the script below.** If it looks good, run the next cell to continue. If not, re-run Cell 2 or edit the script file directly.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

# Display the script
script_path = results.get('script', '')
if script_path and Path(script_path).exists():
    script_text = Path(script_path).read_text()
    display(Markdown(script_text))
else:
    print('⚠️  Script file not found')

print('\n' + '='*60)
print('👆 Review the script above.')
print('   If approved, run the next cell to continue.')
print('   To regenerate, re-run Cell 2.')

---
## Cell 4: Generate Voice + Scenes + Assemble

⚡ This cell does the heavy lifting: voice generation, image generation, and video assembly.

In [ ]:
from src.pipeline import continue_after_script_review

#=== TTS Settings ===#
TTS_ENGINE = 'fish'         # 'fish', 'qwen_1.7b', or 'qwen_0.6b'
REFERENCE_AUDIO = None      # Path to voice sample (for Qwen cloning)
BGM_PATH = None             # Path to background music (or None)
BGM_VOLUME = 0.15           # Background music volume (0-1)
KEN_BURNS = True            # Apply zoom/pan effect on scenes

if SHEET_ROW:
    update_status(SHEET_ROW, 'generating')

results = continue_after_script_review(
    results=results,
    tts_engine=TTS_ENGINE,
    reference_audio=REFERENCE_AUDIO,
    bgm_path=BGM_PATH,
    bgm_volume=BGM_VOLUME,
    ken_burns=KEN_BURNS,
)

---
## ⏸️ Cell 5: REVIEW VIDEO

**Watch the video below.** If it looks good, run the final cell to save everything.

In [ ]:
from IPython.display import Video, display, HTML
from pathlib import Path

video_path = results.get('video', '')
if video_path and Path(video_path).exists():
    print(f'🎬 Video: {video_path}')
    print(f'   Duration: {results.get("total_duration", 0):.1f}s')
    print(f'   Scenes: {results.get("scene_count", 0)}')
    
    # Display video player
    try:
        display(Video(video_path, embed=True, width=800))
    except Exception:
        print(f'   (Preview not available — check the file at {video_path})')
else:
    print('⚠️  Video file not found')

# Show thumbnails
thumbs = results.get('thumbnails', [])
if thumbs:
    print(f'\n🖼️ Thumbnail variants ({len(thumbs)}):')
    from IPython.display import Image
    for tp in thumbs:
        if Path(tp).exists():
            display(Image(filename=tp, width=400))

# Show SEO
seo = results.get('seo', {})
if seo:
    print(f'\n📊 SEO Metadata:')
    print(f'   Title: {seo.get("title", "N/A")}')
    print(f'   Tags: {", ".join(seo.get("tags", [])[:8])}...')

print('\n' + '='*60)
print('👆 Review the video, thumbnails, and SEO above.')
print('   If approved, run the next cell to finalize.')

---
## Cell 6: Finalize & Save to Drive

In [ ]:
# Update Google Sheet with results
if SHEET_ROW:
    drive_link = results.get('video', '')
    update_status(
        SHEET_ROW, 
        'complete',
        extra_data={'drive_link': drive_link}
    )

print('🎉 Video production complete!')
print(f'   📁 Project folder: {results.get("project_dir", "")}')
print(f'   🎬 Video: {results.get("video", "")}')
print(f'   🖼️  Thumbnails: {len(results.get("thumbnails", []))} variants')
print(f'   📊 SEO: {results.get("seo", {}).get("title", "N/A")}')
print()
print('📌 Next steps:')
print('   1. Download video from Google Drive')
print('   2. Upload to YouTube (or let n8n handle it)')
print('   3. Set thumbnail')
print('   4. Paste SEO title/description/tags')
print()
print('🔄 To make another video, go back to Cell 1!')

---
## 🔧 Utilities

In [ ]:
# === UTILITY: Clone your voice on Fish Audio ===
# Run this ONCE with your voice sample, then save the voice_id

# from src.voice import clone_voice_fish
# voice_id = clone_voice_fish(
#     audio_path='/content/drive/MyDrive/my_voice_sample.wav',
#     voice_name='my-narrator-voice',
# )
# print(f'Set this in your config: FISH_VOICE_ID={voice_id}')

In [ ]:
# === UTILITY: Initialize Google Sheet template ===
# Run this ONCE to set up your content calendar headers

# from src.sheets import create_sheet_template
# create_sheet_template()